# Event Definition Notebook

This notebook guides you through the process of defining, analyzing, and visualizing a extreme metereological event.

---

## How to Use This Notebook

**1. Follow the numbered steps in order.**  
Each section builds upon the previous one, from setup, data loading, and climatology computation, to event analysis and visualization.

**2. Look for <font color="orange"> Orange cells  </font> and code cells marked as <font color="lightgreen">##### (User selection) ##### </font>:** 
| <font color="orange"> Orange cells  </font> | <font color="orange"> Need user intervention </font>|
| ----------- | ----------- |
| <font color="green">**Green cells** </font> | <font color="green">**Run automatically on user input provided in the orange cells and should not be adjusted in most cases** </font>|


**3. Run cells sequentially.**  
Start from the top and execute each cell (`Shift` + `Enter`).  



## Setup and User Inputs
This section prepares the environment for running the analysis.


### <font color="green"> Import require packages </font>

In [ ]:
from c3s_event_attribution_tools import *
import json
from shapely.ops import unary_union
import time
import calendar
import matplotlib.colors as colors
import matplotlib

### <font color="orange"> User specifications </font>

#### <font color="orange"> Authentication & File Setup </font>

Action Required: Ensure you have entered your **CDS API Key** in the code cell above.

Storage: Data and results will be saved to: {{your_save_directory}}.

Security: ⚠️ Never share this notebook publicly or commit it to GitHub with your API keys visible.

In [ ]:
# Api key used for the DataClient, replace with your own API key from the C3S Climate Data Store (https://cds.climate.copernicus.eu/profile) 
################# (User selection) ###################
your_api_key = ' ' 
######################################################

# Do not touch
CURRENT_DIRECTORY = os.getcwd() 

# Directory you wish to store output files in
################# (User selection) ###################
your_save_directory = os.path.abspath(os.path.join(CURRENT_DIRECTORY, "./data"))   # change ./data to your desired directory
######################################################

# Create directory if it does not exist
os.makedirs(your_save_directory, exist_ok=True) 

## <font color="orange"> 2.1 Choice of parameter </font>

In [ ]:
# Choice of parameter (Tmax, Tmean, Tmin, Precipitation)
################# (User selection) ###################
parameter = " "  
######################################################

# Get parameter configuration
config = Utils.get_parameter_config(parameter)
variable = config["variable"]
value_col = config["value_col"]
y_label = config["y_label"]
unit = config["unit"]
calculation = config["calculation"] 
method = config["method"] 
datetime_col = config["datetime_col"]
from_unit = config.get("from_unit")
to_unit= config.get("to_unit")
era5_p = Utils.var_map(parameter, "era5")
variable_name = getattr(Variable.ERA5DailySingleLevel, era5_p)

## <font color="orange"> 2.2 Choice of area </font>

In [ ]:
################# (User selection) ###################
# Area of interest bounding box:
bbox = (, , , ) # (Southern boundary, Western boundary, Northern boundary, Eastern boundary)

# Date of the event 
event_end = datetime(, , ) #YYYY,MM,DD
######################################################

# Start date of 14 days leading up to the event
event_start = event_end - timedelta(days=14)
bbox = Utils.convert_bbox(*bbox)

## 2.2.a.i Prepare Climate Data and Visualize event daily maps
In this step, you will download daily data for the selected climate parameter (e.g., temperature, precipitation) 
from the **ERA5/Beacon** dataset. The data will be divided into two key time periods:

- **Event period** - the days leading up to and including the event.  
- **Climatology period** - a long-term reference window (typically 1991–2020).

Finally, you will visualize the event evolution over time with daily maps.

### <font color="green"> Download the Daily Dataset for the Chosen Variable </font>
This cell connects to the CDS and ERA5/Beacon API and retrieves daily gridded data for your area of interest, from 1950 up to the event date.

In [ ]:
# Connection to the DataClient
client = DataClient(cds_key=your_api_key, beacon_cache_url="https://beacon-era5.maris.nl/")

# Run code to fetch data using beacon (Tmax, Tmean, Tmin, Precipitation)
gr_daily_xr = client.fetch_era5_daily_single_levels_xr(variable=variable_name, bbox = bbox, time_ranges=[(datetime(1950, 1, 1), event_end)],  from_unit = from_unit, to_unit = to_unit)

### <font color="green"> Define the Event Period and Climatology Period datasets </font>
The dataset is divided into two parts: the event period and the long-term climatology period (e.g., 1991–2020).

In [ ]:
# Specify climatology dates 
clim_start = datetime(1991, 1, 1)
clim_end = datetime(2020, 12, 31)

# Select event subset
gr_daily_eventNdays_xr = gr_daily_xr.sel(valid_time=slice(event_start, event_end))

# Select climatology subset
gr_daily_clim_xr = gr_daily_xr.sel(valid_time=slice(clim_start, clim_end))

### <font color="green"> Visualize Daily Maps Before and During the Event </font>
A sequence of daily maps is produced to show how the variable evolved in the 15 days up until the event.

In [ ]:
gr_daily_eventNdays = gpd.GeoDataFrame(gr_daily_eventNdays_xr.to_dataframe().reset_index(),
    geometry=gpd.points_from_xy(gr_daily_eventNdays_xr.to_dataframe().reset_index().longitude, gr_daily_eventNdays_xr.to_dataframe().reset_index().latitude),
    crs="EPSG:4326"
)

# Specifying the title
title = f'Daily {variable} ({unit})'

fig, axes, img_ax = Plot.subplot_gdf(gr_daily_eventNdays, datetime_col="valid_time", value_col=value_col, ncols=5, legend_title=title)

## 2.2.a.ii Compute and Visualize Event Anomalies
In this step, we compare the observed event with the long-term conditions (Event Anomaly).


### <font color="green"> Compute the 31-Day Running Mean Climatology (1991–2020) </font>
This step calculates the smoothed climatology for each day of the year, using a 31-day window (±15 days).
It averages all years between 1991 and 2020 to create a stable reference dataset. Additionally, we extract the event period from the smoothed climatology

In [ ]:
clim31d_xr = Process.sliding_stat_by_dayofyear(gr_daily_clim_xr, pad=15, method='mean')
clim31d_eventNdays_xr = clim31d_xr.sel(dayofyear=slice(event_start.timetuple().tm_yday, event_end.timetuple().tm_yday))

### <font color="green"> Save the Climatology Dataset </font>
Once the climatology is computed, it is saved as a NetCDF (.nc) file.
This ensures you can reuse the climatology later without recalculating it.

In [ ]:
# Define file name and save path
climatology_save = f"climatology_1991-2020_{variable}"
save_path = os.path.join(your_save_directory, f"{climatology_save}.nc")

# Save to NetCDF
clim31d_xr.to_netcdf(save_path)

# If precipitation chosen as variable & we are in April or later in the year, we retrieve also the temperature climatology needed for the trend analysis step
if variable == "Precipitation" and datetime.now().month > 3:
    save_path = os.path.join(your_save_directory, f"climatology_1991-2020_Temperature.nc")
    temperature_daily = client.fetch_era5_daily_single_levels_xr(variable=Variable.ERA5DailySingleLevel.temperature_2m_mean, bbox = bbox, time_ranges=[(datetime(1950, 1, 1), event_end)],  from_unit = "k", to_unit = "c")
    temperature_daily_clim = temperature_daily.sel(valid_time=slice(clim_start, clim_end))
    clim31d_temp = Process.sliding_stat_by_dayofyear(temperature_daily_clim, pad=15, method='mean')
    clim31d_temp.to_netcdf(save_path)

### <font color="Green"> Visualize Daily Anomalies During the Event </font>
Finally, we calculate and plot the absolute anomaly for each day of the event period, in case the of precipitation, the relative anomaly is calculated:

$Anomaly=Event Value−Climatology Value$

$Relative Anomaly=(Event Value−Climatology Value)/Climatology  Value$

The output shows 15 daily maps, allowing users to visualize how the (relative) anomaly evolved in space and time.

In [ ]:
clim31d_eventNdays = gpd.GeoDataFrame(
    clim31d_eventNdays_xr.to_dataframe().reset_index(),
    geometry=gpd.points_from_xy(clim31d_eventNdays_xr.to_dataframe().reset_index().longitude, clim31d_eventNdays_xr.to_dataframe().reset_index().latitude),
    crs="EPSG:4326"
)

# Converting day of year to dates used for plots.
clim31d_eventNdays['valid_time'] = pd.to_datetime(event_end.year * 1000 + clim31d_eventNdays["dayofyear"], format="%Y%j")

# Specifying the title
if parameter in ["Tmax", "Tmean", "Tmin"]:
    title=f'Daily {variable} Anomaly ({unit})'
else:
    title=f'Daily {variable} Anomaly (-)'

# Calculate the anomaly for the days leading up to the event
gr_anom_eventNdays = Process.calculate_anomaly(event_gdf=gr_daily_eventNdays, mean_climatology_gdf=clim31d_eventNdays, value_col=value_col, datetime_col="valid_time", calcation=calculation)

fig, axes, img_ax = Plot.subplot_gdf(gr_anom_eventNdays,datetime_col="valid_time", value_col=value_col, ncols=5, legend_title=title, cmap='anomaly')

## <font color="orange"> 2.2.b Choose a Day for Spatial Overlays (Steps 2.c–2.g) </font>
Pick a single representative day during the event to use for interactive region selection and inspection. This single-day snapshot is used to visually choose and test region overlays (e.g., river basins or administrative regions) that will be used for the spatial event definition.

In [ ]:
############ Selected date (User selection) ##########
selected_date = datetime(, , ) #YYYY,MM,DD
######################################################

### <font color='green'>Optionally select different or more overlays in addition to those pre-set below</font>


In [ ]:
# Overlay images
anomaly_img = Utils.get_base_fig(selected_date, gr_anom_eventNdays, value_col=value_col) # Anomaly
event_img = Utils.get_base_fig(selected_date, gr_daily_eventNdays, value_col=value_col)  # Event
overlays = {
    "anomaly": anomaly_img,
    "event": event_img
}

## <font color="green"> 2.2.c & 2.2.d Define and Select the Event Region </font>
In this step, you will visually inspect the event map and select one or more regions that represent the spatial extent of the event.
The goal is to define a realistic event region, large enough to capture the full impact area.
Regions are selected interactively in the browser using the region picker.

### Make sure to click on the link in the cell output to open the region picker!

Notes:
- To avoid selection bias (maximising the extremity of the event) the regions selected should be a larger area that encompasses the main event rather than selecting only the area of maximum intensity.

In [ ]:
# Regiontype wraf for temperature, hydrobasin for precipitation
if parameter in ["Tmax", "Tmean", "Tmin"]:
    region_type = 'wraf'
elif parameter == "Precipitation":
    region_type = 'hydrobasin'

result = Utils.select_region(regionType=region_type, bbox=bbox, overlays=overlays) 
data = json.loads(result)

polygons, coords = Utils.data_2_poly(data)

elevation_nc = xr.open_dataset("./00_BASE_DATA/elev.0.25-deg.nc")
elevation = Utils.wrap_lon(elevation_nc)['data']

fig, ax = Plot.plot_poly(polygons, coords, layer=elevation)

## <font color="orange"> 2.2.e Apply an Elevation Threshold </font>

In this step, you can refine the event region by removing areas above a certain elevation.
This is useful when your region includes mountains or strong elevation gradients that could distort averages (for example, separating coastal plains from nearby highlands).

The tool will overlay topography on your selected event region so you can decide whether to apply an elevation cutoff.

Notes:
- Run this cell as much as needed to ensure the region is topographically homogeneous.

In [ ]:
################### Select an elevation threshold (User selection)  ########################
elevation_threshold = 3000 # meters, the studyregion is selected with all areas below the threshold
############################################################################################

# The adjusted_polygons are later used for visualizing the studyregion in the plots and for slicing out the data only for that region
fig, ax, adjusted_polygons = Plot.elevation_region(data, polygons, elevation, elevation_threshold)

## <font color="orange"> 2.2.e Refine Region by Köppen–Geiger Climate Classification </font>
 
In this step, you can refine the selected event region by excluding specific climate zones based on the Köppen–Geiger classification.
This helps to isolate areas that share similar climatic characteristics, ensuring that subsequent analyses (e.g., temperature or precipitation trends) are climatologically consistent.
 
The tool overlays the Köppen–Geiger map on top of your selected region, allowing you to:
 
Visually inspect which climate classes dominate the area.
 
Filter polygons to remove specific classes (e.g., “Tropical”, “Arid”, “Temperate”, “Cold”, “Polar”).
 
Subclasses (e.g., "Dfc", "Cfa", "Cfb", "ET", "EF") are also possible to filter out. Run the cell once to obtain the region plot and from the legend you can obtain the subclasses codes
 
Notes:
 
The Köppen–Geiger classification is derived from Beck et al. (2023), providing high-resolution (1 km) climate zones.
 
This step ensures that your event region represents climate-homogeneous conditions, improving the interpretability of metrics like mean temperature or accumulated precipitation.
 
Run this step multiple times if needed to fine-tune the selected area.

In [ ]:
############## User Selection ####################
# Select the Köppen-Geiger classes (Classes available: "Tropical", "Arid", "Temperate", "Cold", "Polar") or subclasses (e.g., "Dfc", "Cfa", "Cfb", "ET", "EF") to filter out. You can check the subclasses codes from the plot legend
classes = []
##################################################
 
# Load Köppen–Geiger data
kg = xr.open_dataset("./00_BASE_DATA/koppen_geiger_0p1.nc")
kg = Utils.wrap_lon(kg)
kg_da = kg["kg_class"]
legend_path = os.path.expanduser("./00_BASE_DATA/legend.txt")
 
# Filter polygons by Köppen-Geiger classes
filtered_polys = Process.filter_polygons_by_kg(adjusted_polygons, kg_da, classes, invert=True)
 
# Plot
fig, ax = Plot.plot_koppen_geiger(kg_da, filtered_polys, coords, legend_path, extra_polygons=adjusted_polygons)

## <font color="green"> 2.3  Save the study domain </font>

This step saves the final event region (also called the study domain) that you defined in the previous steps.


In [ ]:
shapefile_save = "sf_studyregion"

# The studyregion is used for masking the geospatial data
multipoly = unary_union(filtered_polys)
studyregion = gpd.GeoDataFrame(index=[0], crs='EPSG:4326', geometry=[multipoly])    
studyregion.to_file(os.path.join(your_save_directory, f"{shapefile_save}.shp"))

## <font color="green"> 2.4 Create and Save the Daily Time Series </font>

In this step, the daily values of the chosen climate variable (e.g., temperature, precipitation) are averaged over the study domain you defined earlier.
This produces a single daily time series representing the evolution of the event across the region. The time series is saved and it will be reuse in further steps.

In [ ]:
# Spatial Masking
mask = regionmask.mask_geopandas(studyregion, gr_daily_xr.longitude, gr_daily_xr.latitude)
ts_regional = gr_daily_xr.where(mask == 0, drop=True)

# Regional Time Series (Latitude Weighted)
weights = Process.weighted_values(ts_regional, value_col=None, lat_col='latitude')
ts_daily_studyregion_xr = ts_regional.weighted(weights).mean(['longitude', 'latitude']).sortby("valid_time")         

# Specify name for saving the daily timeseries 
daily_timeseries_save = "ts_daily_studyregion"
ts_daily_studyregion_xr.to_netcdf(f"{your_save_directory}/{daily_timeseries_save}.nc")

## 2.5 Calculate and Compare n-Day Aggregated Values

This step aggregates  the daily data into n-day averages or accumulations to analyze how the event compares to the historical annual time series in the study region. (n-day accumulations for precipitation and n-day averages for temperature)
It helps to evaluate the event’s intensity, duration, and persistence at standard time scales.

Comparing these values to the hsitorical events helps identify whether the event was exceptional or within normal variability.

### <font color='green'>Select Parameters</font>

This cell automatically configures the n-day aggregation settings based on the variable you’re analyzing:
- **Precipitation:** uses 1-, 3-, 5-, and 10-day totals.  
- **Temperature:** uses 1-, 3-, 7-, and 14-day averages.  

You can adjust optional parameters (e.g., centering or quantile) if needed. Otherwise, the standard settings are given. 
Run the cell below to confirm your settings before continuing.

In [ ]:
############# (Optional parameters) #############
# method: mean, sum, std, quantile
method = None
centering = True
# if method quantile is selected
quantile = .9

###########################################

if parameter in ["Tmax", "Tmean", "Tmin"]:
    days = [1, 3, 7, 14]
    if method == None:
        method = 'mean'
elif parameter == "Precipitation":
    days = [1, 3, 5, 10]
    if method == None:
        method = 'sum'

if method == 'mean':
    title = f"Average {parameter} ({unit})"
elif method == 'sum':
    title = f"Total {parameter} ({unit})"
elif method == 'std':
    title = f'{parameter} variability ({unit})'
elif method == 'quantile':
    title = f'{parameter} {int(quantile*100)}th Percentile ({unit})'

### <font color="green"> Compute and Plot n-Day Averages or Accumulations </font>

The code below computes the n-day moving average (for temperature) or accumulation (for precipitation) and plots each time series for comparison.
These plots help visualize how each timescale smooths or amplifies the event’s intensity.

#### The yellow markings indicate the 15 days leading up to the event last day.

In [ ]:
ts_daily_studyregion = ts_daily_studyregion_xr.to_dataframe().reset_index() 

# Calculate n-day rolling values
rolled_data_list = [Process.calculate_rolling_n_days(gdf=ts_daily_studyregion,
                                            value_col=value_col, datetime_col=datetime_col,
                                            padding=d, centering=centering, method=method, quantile=quantile) for d in days]

# Labels and label ticks
labelticks = pd.date_range("2000-01-01", "2000-12-31", freq="MS").dayofyear
labels = pd.date_range("2000-01-01", "2000-12-31", freq="MS").strftime("%b")

# Plot timeseries
fig, axes, img_ax = Plot.plot_n_days(rolled_data_list, value_col=value_col, parameter=parameter,
                                event_date=event_end, labelticks=labelticks, labels=labels,
                                days=days, title=title, fig_height=4, ncols=2)

## <font color='green'> 2.6 Analyze, Plot and save the Seasonal Cycle (1991–2020) </font>

The seasonal cycle represents how the chosen climate variable typically evolves over the course of a year, based on the long-term climatology (1991–2020). Two plots are shown to account for the fact that the timing of summer and winter seasons differs between the Northern and Southern Hemispheres

In [ ]:
clim31d = gpd.GeoDataFrame(
    clim31d_xr.to_dataframe().reset_index(),
    geometry=gpd.points_from_xy(clim31d_xr.to_dataframe().reset_index().longitude, clim31d_xr.to_dataframe().reset_index().latitude),
    crs="EPSG:4326"
)

clim31d = clim31d.rename(columns={"dayofyear": "doy"})
clim31d['valid_time'] = pd.to_datetime(event_end.year * 1000 + clim31d["doy"], format="%Y%j")

# Compute both seasonal cycles
ts_clim31d_studyregion, plot_df1, labels1, labelticks1 = Process.calculate_seasonal_cycle(clim31d, studyregion, value_col=value_col, event_end=event_end, datetime_col=datetime_col, month_range=(1, 12))
ts_clim31d_studyregion2, plot_df2, labels2, labelticks2 = Process.calculate_seasonal_cycle(clim31d, studyregion, value_col=value_col, event_end=event_end, datetime_col=datetime_col, month_range=(7, 6))

In [ ]:
# Calculating the annual climatology (1991-2020) time series for the studyregion and plotting the seasonal cycles
y_label = unit
title1 = f"Seasonal Cycle {parameter} Jan–Dec"
title2 = f"Seasonal Cycle {parameter} Jul–Jun"

figs, axes = plt.subplots(1, 2, figsize=(14, 8))

fig, ax, img_ax = Plot.plot_timeserie(data=plot_df1, value_col=value_col, datetime_col="plot_time", title=title1,
        x_label="Date", y_label=y_label, line_style=":", labels=labels1,
        labelticks=labelticks1, add_logos=False, ax=axes[0]
    )

fig, ax, img_ax = Plot.plot_timeserie(data=plot_df2, value_col=value_col, datetime_col="plot_time", title=title2,
    x_label="Date", y_label=y_label, line_style=":", labels=labels2,
    labelticks=labelticks2, add_logos=True, ax=axes[1]
)

axes[1].set_xlim(plot_df2["plot_time"].min(), plot_df2["plot_time"].max())

### <font color="green"> Save the Seasonal Cycle Outputs </font>
Save the computed seasonal cycle data and the plot for future reference.

In [ ]:
################# User Selection  ##################
# Please specify names for saving the dataframe and figure 
# Time series file name
seasonal_cycle_series_save = 'seasonal_cycle_1991-2020'

# Figure file name
seasonal_cycle_figure_save = 'seasonal_cycle_1991-2020'
#####################################################

# Save the time series
ds = xr.Dataset.from_dataframe(ts_clim31d_studyregion.set_index(datetime_col))
ds.to_netcdf(os.path.join(your_save_directory, f"{seasonal_cycle_series_save}.nc"))

# Save the figure
figs.savefig(fname=os.path.join(your_save_directory, f"{seasonal_cycle_figure_save}.png"), dpi=150, bbox_inches="tight")

## 2.7 Decide on the temporal extent  
Use **Step 2.2** and the following considerations:

#### a. As much related to impacts as possible
- e.g. `TXx` (max Temperature 1-day, outdoor workers)  
- e.g. `T3x` (mean Temperature 3-day, people indoors)  
- e.g. Flooding: precipitation averaged over duration or over response time basin  
- e.g. Drought: precipitation values (or soil moisture) averaged over several months or multiple rainy seasons  

#### b. Seasonality
- **OPTIONAL:** Use information from literature review (see also Sec. 6a.), specifically for information on seasonality, that can feed into the choice of event definition.  
- Make use of seasonal cycle plot  
- Restrict to some months if necessary, e.g. if event occurred outside of the usual season or a change in seasonality is suspected  
  - Example: May–June maximum 4-day precipitation  
- Potentially take temperature anomalies rather than absolute values if averaging over months where the seasonal cycle is in transition between peaks and troughs  

#### c. Make use of the plots created in Step 2.5

## 2.8 Make a final decision on the event definition  
Update the output table in the tables document (apart from the last row on return period).

#### a. Write down in the cell below the factors that informed the specific event definition  
Use the following considerations as a guide to single out which aspect of the multifaceted nature of the extreme is chosen for emphasis and communication:

- **Variable:**  
  - For temperature: (`Tmax` or `Tmin` or `Tmean` – were the daily maxima or the high nighttime minima or both the most defining for the heat event?) 

- **Timing:**  
  - Is it the early-season onset of the event that made it anomalous (only) for the time of year, giving locals little time to adjust?  

- **Seasonality:**  
  - Is it specific for a single season only, in case of multiple peaks in the seasonal cycle?  

- **Duration:**  
  - Is it daily record-breaking extremes that will dominate headlines?  
  - Or the long persistence of heat that will also impact indoor conditions, drought, or fire weather, and be felt more than one day of exceptional heat?  

- **Spatial:**  
  - Is the large spatial extent of the event particularly noteworthy over other factors and in comparison to previous similar events in the region?  
  - Or is it that the event location coincides with a densely populated area that makes it particularly noteworthy?

### Write down your decisions here
*
*
*
*

## 2.9 Create and Save the Annual Time Series
This step creates an annual time series. For each year in the dataset, one representative value is calculated, using the following input variables:

- yearly_value: Choose either the `max`, `mean` or `min` value for the selected time period.
- padding: Choose the number of days used for your rolling window.
- month range: Choose the date range, this can be one month, multiple months and also december-january (over the end of the year)
- method: Default uses the rolling `mean` for temperature and rolling `sum` for precipitation. Can be overwritten.

How does an example input look? 

If user is looking at `Tmax` and chooses yearly_value = `max`, padding = 5, month_range = (5,6), method = None. For each year the following value will be calculated:
* For the months May-June, a 5 day-rolling `mean` of `Tmax` will be calculated. Of these rolling-means for each year the `max` value will be taken, which is the annual value being used.

This allows for long-term comparison of how the event intensity varies from year to year.

### <font color='orange'>Select Parameters</font>

You can adjust optional parameters if needed. Run the cell below to confirm your settings before continuing.

In [ ]:
################### User Selection  ##################
# Choose mean, max, min
yearly_value = ''

# Padding >= 1 : rolling window, n-days (centered)
padding = 1

# Choose month range, e.g. (1, 12) or (12, 1), for one month e.g. January please input (1, 1)
month_range = (,)

# Standard method for temperature: mean and for precipitation: sum, if not specified, if you want to change it, specify here
method = None
#######################################################

#### <font color='green'>Generate plotting details (will be transferred to back-end library later)</font>

In [ ]:
# Automatically generate month range label and title
start_month_name = calendar.month_abbr[month_range[0]]
end_month_name = calendar.month_abbr[month_range[1]]

# Handle different types of ranges
if month_range[0] < month_range[1] or month_range[0] == month_range[1]:
    # Normal within-year range (e.g., May–June)
    month_label = f"{start_month_name}–{end_month_name}"
else:
    # Cross-year range (e.g., Dec–Feb)
    month_label = f"{start_month_name}–{end_month_name} (cross-year)"

y_label = unit

line_style='-'  

draw_style='steps'

if parameter in ["Tmax", "Tmean", "Tmin"]:
    if yearly_value == 'max' or yearly_value == 'min':
        title = f"{padding}-day Rolling {yearly_value.capitalize()}imum of {parameter.capitalize()} ({unit}) ({month_label})"
    elif yearly_value == 'mean':
        title = f"{padding}-day Rolling {yearly_value.capitalize()} of {parameter.capitalize()} ({unit}) ({month_label})"
    if method is None:
        method = 'mean'
elif parameter == "Precipitation":
    title = f"{padding}-day Rolling {yearly_value.capitalize()} Total {parameter.capitalize()} ({unit}) ({month_label})"
    if method is None:
        method = 'sum'

#### <font color="green"> Calculation and plotting </font>
The script computes one value per year, then plots it.
This visualization helps identify long-term changes, extremes, or trends in the event’s intensity.

In [ ]:
# Calculating the yearly value based on input (e.g. 3 day max temperature in months 1-12)
ts_ann_studyregion, rolled_gdf = Process.calculate_yearly_value(gdf=ts_daily_studyregion,
                                            value_col=value_col, datetime_col=datetime_col,
                                            yearly_value=yearly_value, padding=padding,
                                            month_range=month_range, method=method)

# Plot annual time series
fig, ax, img_ax = Plot.plot_timeserie(data=ts_ann_studyregion, value_col=value_col,
                                 datetime_col='year', title=title, x_label='Date',
                                 y_label=y_label, line_style=line_style, draw_style=draw_style)

### <font color='green'> Saving output </font>
Save the resulting annual time series as a NetCDF file for analysis and data exchange

In [ ]:
################### User Selection  ##################
# Please specify name for saving the annual time series
annual_timeseries_save = 'ts_ann_studyregion'
#######################################################

ds = xr.Dataset.from_dataframe(ts_ann_studyregion.set_index("year"))
ds.to_netcdf(os.path.join(your_save_directory, f"{annual_timeseries_save}.nc"))

## 2.10 Read in station data time series from a CSV file

### <font color='orange'>Optionally load in data from a csv or xls file</font>

In [ ]:
# CSV
# separator = ','
# encoding = 'utf-8'
# file_name = '../data/my_file.csv'   
# df = pd.read_csv(file_name, sep=separator, encoding=encoding)

In [ ]:
# XLS
# xls_station_data = pd.ExcelFile("../data/example.xlsx")
# print(xls_station_data.sheet_names)   # list available sheets
# xls_station_data_sheet1 = pd.read_excel(xls_station_data, "Sheet1")
# xls_station_data_sheet2 = pd.read_excel(xls_station_data, "Sheet2")

## 2.11 Create Event Map

This steps produce a figure of the event (Averaged over the selected temporal event definition). Additionally, the event anomaly map is produced as well and an overlay of the study region is included.

Please, select the selected date based on the next table where the top 5 highest values for the chosen date range using the padding defined in 2.9 are shown. Additionally, choose a padding corresponding to the temporal event definition. 

Note that the rolling date is indicated as follows:
* For uneven rolling windows, the reported date corresponds to the middle date. For example in a 7-day rolling window, the 4th day is reported in the table.
* For even rolling windows, the reported date corresponds to the day immediately to the right of the midpoint. For example, in a 4-day rolling window, the 3rd day is reported in the table. 

In [ ]:
# Top 5 highest values for the event year using the padding defined in 2.9
rolled_gdf.loc[rolled_gdf['year'] == rolled_gdf['year'].max()].sort_values(by=value_col, ascending=False).head(5)

In [ ]:
# Max 1-day values, using days that appear in the last 15 days of the time series
ts_daily_studyregion.tail(15).sort_values(by=value_col, ascending=False).head(5)

### <font color='orange'>Parameters</font>

You can adjust optional parameters if needed. Run the cell below to confirm your settings before continuing.

In [ ]:
################### User Selection  ##################
# Possible free user input for the chosen date + padding, else it uses the date where the highest annual value was found for the event year
selected_date = pd.to_datetime('--') #YYYY-MM-DD

padding = 1 # Padding >= 1 : window, n-days (centered)
#######################################################

# Event colors
event_poly_col = 'cyan'
anomaly_poly_col = 'lime'

if parameter in ["Tmax", "Tmean", "Tmin"]:
    event_title = f'Mean {parameter}'
    anomaly_title = f'Mean {parameter} Anomaly'
    y_label = '(°C)'
    y_label_anom = '(°C)'
elif parameter == "Precipitation":
    event_title = 'Total Precipitation'
    anomaly_title = 'Total Precipitation Anomaly'
    y_label = '(mm)'
    y_label_anom = '(-)'

### <font color="Green"> Calculation and plotting</font>

In [ ]:
# If selected_date is None, it uses the highest annual value found for the event year
selected_date = selected_date if selected_date else pd.to_datetime(ts_ann_studyregion.iloc[-1, 1])

# Add padding around the selected date
half_window = (padding - 1) / 2
selected_date_start = selected_date - timedelta(days=half_window)
print("selected date start:", selected_date_start)
selected_date_end = selected_date + timedelta(days=half_window)
print("selected date end:", selected_date_end)

# Format date range string
if selected_date_start.date() == selected_date_end.date():
    date_range_str = selected_date.strftime('%Y-%m-%d')
else:
    date_range_str = f"{selected_date_start.strftime('%Y-%m-%d')} to {selected_date_end.strftime('%Y-%m-%d')}"

# Event selection (this assumes the selected_date falls inside the 15 days up until the event)
selected_intersect = Utils.subset_gdf(gdf=gr_daily_eventNdays, datetime_col=datetime_col,
                                date_range=(selected_date_start, selected_date_end))

# Anomaly selection (this assumes the selected_date falls inside the 15 days up until the event) 
selected_intersect_anomaly = Utils.subset_gdf(gdf=gr_anom_eventNdays, datetime_col=datetime_col,
                                        date_range=(selected_date_start, selected_date_end))

# Calculate average over the date range
selected_event_mean = Process.calculate_mean(gdf=selected_intersect, value_col=value_col, groupby_col=['longitude', 'latitude', 'geometry'])
selected_anomaly_mean = Process.calculate_mean(gdf=selected_intersect_anomaly, value_col=value_col, groupby_col=['longitude', 'latitude', 'geometry'])

# Create subplot
figs, axes = plt.subplots(1, 2, figsize=(14, 6), subplot_kw={"projection": ccrs.PlateCarree()})

# Plot event
fig, ax, img_ax = Plot.plot_gdf(selected_event_mean, value_col=value_col, ax=axes[0],
                     title=f"{event_title}\n({date_range_str})",
                     legend_title=y_label, add_logos=False, polygons=adjusted_polygons, polygon_color=event_poly_col)
# Plot anomaly
fig, ax, img_ax = Plot.plot_gdf(selected_anomaly_mean, value_col=value_col, ax=axes[1],
                     title=f"{anomaly_title}\n({date_range_str})",
                     legend_title=y_label_anom, add_logos=False, polygons=adjusted_polygons, polygon_color=anomaly_poly_col, cmap='anomaly')


### <font color="green">Save as NetCDF</font>

In [ ]:
################### User Selection  ##################
# Please specify names for saving the dataframe and figure
selected_event_mean_save = f"{variable}_{date_range_str}"

selected_anomaly_mean_save = f"{variable}_anomaly{date_range_str}"
#######################################################

# Drop geometry column to be able to save as NetCDF
df = selected_event_mean.drop(columns="geometry")

# Convert to xarray Dataset
ds = xr.Dataset.from_dataframe(df)

ds.to_netcdf(os.path.join(your_save_directory, f"{selected_event_mean_save}.nc"))

# Drop geometry column to be able to save as NetCDF
df = selected_anomaly_mean.drop(columns="geometry")

# Convert to xarray Dataset
ds = xr.Dataset.from_dataframe(df)
ds.to_netcdf(os.path.join(your_save_directory, f"{selected_anomaly_mean_save}.nc"))

## 2.12 Write a paragraph on the event definition in the scientific report Section 1.

## 2.13 Meteorological maps including the event.

Z500 (500 hPa Geopotential Height) – shows the height of the 500 hPa pressure surface (~5–6 km). It reveals mid-tropospheric circulation: high values mark warm ridges and low values mark cold troughs that steer large-scale weather patterns.

SLP (Sea-Level Pressure) – displays surface pressure distribution (in hPa). Low-pressure systems (cyclones) and high-pressure systems (anticyclones) control winds, fronts, and rainfall patterns near the surface.

### <font color='orange'>Select correlation domain</font>

Choose bounding box for the Z500 and SLP contour plots. This synoptic_domain should be the starting point for the correlation domain to be chosen in the analogues.

In [ ]:
################### User Selection  ##################
synoptic_domain = (, , , ) # (Western boundary, Southern boundary, Eastern boundary, Northern boundary)
#######################################################

### <font color='green'>Parameters, countours and plotting</font>

In [ ]:
if value_col == 't2m':
    contour = 'z500'
    contour_col = 'z'
    legend_title = 'ERA5 Z500 and T2M'
    contour_steps = 2
    conversion = 100
    # Get contour data
    contour_gdf_xr = client.fetch_era5_daily_pressure_levels_xr(variable=Variable.ERA5DailyPressureLevels.geopotential, bbox=synoptic_domain, levels=[500], time_ranges=[[event_start, event_end]])
elif value_col == 'tp':
    contour = 'SLP'
    contour_col = 'msl'
    legend_title = 'Total Precipitation and Sea Level Pressure contours'
    contour_steps = 1
    conversion = 100 # to convert from Pa to hPa
    contour_gdf_xr = client.fetch_era5_daily_single_levels_xr(variable=Variable.ERA5DailySingleLevel.mean_sea_level_pressure, bbox=synoptic_domain, time_ranges=[[event_start, event_end]])

gr_event_xr = client.fetch_era5_daily_single_levels_xr(variable=variable_name, bbox = synoptic_domain, time_ranges=[[event_start, event_end]],  from_unit = from_unit, to_unit = to_unit)

gr_event = gpd.GeoDataFrame(gr_event_xr.to_dataframe().reset_index(),
    geometry=gpd.points_from_xy(gr_event_xr.to_dataframe().reset_index().longitude, gr_event_xr.to_dataframe().reset_index().latitude),
    crs="EPSG:4326"
)

# Adjust contour units
contour_gdf_xr[contour_col] = contour_gdf_xr[contour_col] / conversion

# Convert contour data to GeoDataFrame for plotting
contour_gdf = gpd.GeoDataFrame(contour_gdf_xr.to_dataframe().reset_index(), geometry=gpd.points_from_xy(contour_gdf_xr.to_dataframe().reset_index().longitude, contour_gdf_xr.to_dataframe().reset_index().latitude),
crs="EPSG:4326"
)

### <font color='orange'>Select figsize and number of columns to fit your plot</font>

You can overwrite the contour_steps, i.e. make the contours less or more dense. It defaults to 2 dam for temperature and 1 dam for precipitation if contour_steps left untouched.

In [ ]:
################### User Selection  ##################
figsize = (20, 22)
ncols = 5
contour_steps = contour_steps
#######################################################

In [ ]:
fig, ax, img_ax = Plot.subplot_contours(contour_gdf, gr_event, datetime_col=datetime_col,
                       value_col=value_col, contour_col=contour_col, ncols=ncols,
                       contour_steps=contour_steps, legend_title=legend_title, figsize = figsize)

## 2.14 Describe the event and meteorology in the scientific report Introduction

- a. Describe meteorological situation, using maps created in Step 2.13 and the event maps
created in Step 2.11
- b. Use information from seasonality
- c. Placeholder step to include a more elaborate event description by C3S, based on event
monitoring
- d. Placeholder step to include a more elaborate event description by local NMS
- e. If applicable, refer to the more elaborate event descriptions in both the scientific report Introduction and the Event factsheet. 

## 2.15 Average monthly climatology map

Monthly climatology field to be used for the model validation. For temperature or precipitation metrics this is the 1991-2020 mean of the chosen metric over the selected month/season.   

### <font color="green"> Calculation and plotting </font>

In [ ]:
# Automatically selects the month(s) of the event, can be overwritten by user
start_month, end_month = event_start.month, event_end.month
# Handle wrap-around across year end (e.g., Nov–Feb)
if end_month < start_month:
    month_indices = list(range(start_month, 13)) + list(range(1, end_month + 1))
else:
    month_indices = list(range(start_month, end_month + 1))

# Convert to month names
month_names = [calendar.month_name[m] for m in month_indices]

# Define readable label
if len(month_names) == 1:
    period_label = month_names[0]
else:
    period_label = f"{month_names[0]}–{month_names[-1]}"  # e.g., May–June

# Choose appropriate title
if parameter in ["Tmax", "Tmean", "Tmin"]:
    title = f"Mean {parameter} 1991–2020 climatology ({period_label})"
elif parameter == "Precipitation":
    title = f"Mean Total {parameter} 1991–2020 climatology ({period_label})"

# Set unit label
y_label = unit
dpi = 150

# Subset climatology
month_gdf = Utils.subset_gdf(gdf=clim31d, datetime_col=datetime_col, year_range=(event_end.year, event_end.year), month_range=(start_month, end_month)) 

# Calculate average over months
monthly_avg_gdf = Process.calculate_mean(gdf=month_gdf, value_col=value_col, groupby_col=['longitude', 'latitude', 'geometry'])
# Plot
fig, ax, img_ax = Plot.plot_gdf(monthly_avg_gdf, title=title, legend_title=y_label, value_col=value_col, dpi=dpi)


### <font color='green'>Saving the monthly climatology</font>

In [ ]:
################### User Selection  ##################
# Please specify names for saving the dataframe and figure
monthly_avg_gdf_save = f"{variable}_{event_end.month}"

monthly_avg_figure_sace = f"{variable}_{event_end.month}"
#######################################################

# Drop geometry column to be able to save as NetCDF
df = monthly_avg_gdf.drop(columns="geometry")

# Convert to xarray Dataset
ds = xr.Dataset.from_dataframe(df)
ds.to_netcdf(os.path.join(your_save_directory, f"{monthly_avg_gdf_save}.nc"))

fig.savefig(fname=os.path.join(your_save_directory, f"{monthly_avg_figure_sace}.png"), dpi=150, bbox_inches="tight")